# 02 — Hypermobility: Beighton Score Extraction & Classification

Loads the Beighton assessment, extracts scores, classifies participants using age-adjusted thresholds, and merges the `hypermobility` column into `combined_data.csv`.

In [49]:
import pandas as pd

RAW_PATH    = '../data/raw/Hypermobility assessment (Responses) - Form responses 1.csv'
HYPER_PATH  = '../data/processed/hypermobility.csv'
SURVEY_PATH = '../data/processed/combined_data.csv'

raw = pd.read_csv(RAW_PATH)
print('Raw shape:', raw.shape)
print('Columns:', raw.columns.tolist())


Raw shape: (25, 8)
Columns: ['Timestamp', 'Can you bend your  little (pinky) finger back beyond 90° at the knuckle (MCP joint) while keeping the hand flat?\n(Indicate for both LEFT and RIGHT)', 'Can you bend your thumb to touch your forearm (flexor side)?', 'When your arm is fully straight, does your elbow bend backwards more than 10° (hyperextend)?', 'When standing with your knee fully straight, does it bend backwards more than 10° (hyperextend)?', 'With knees straight, can you bend forward and place BOTH palms flat on the floor?', 'Enter your full name', 'Beighton Score']


## 1. Extract name and Beighton score


In [50]:
# Name is the 7th column (index 6), Beighton Score is the 8th (index 7)
df = raw.iloc[:, [6, 7]].copy()
df.columns = ['name', 'beighton_score']

df = df.dropna(subset=['name', 'beighton_score'], how='all')
df['name'] = df['name'].str.strip()
df['beighton_score'] = pd.to_numeric(df['beighton_score'], errors='coerce')

print(f'Participants with data: {len(df)}')
print(f'Missing names: {df["name"].isna().sum()}')
print(f'Missing scores: {df["beighton_score"].isna().sum()}')
df


Participants with data: 25
Missing names: 0
Missing scores: 0


,name,beighton_score
0,P16,1
1,P03,5
2,P24,0
3,P22,1
4,P06,8
5,P12,4
6,P04,5
7,P25,6
8,P18,2
9,P08,4


## 2. Save intermediate hypermobility.csv


In [51]:
df.to_csv(HYPER_PATH, index=False)
print(f'Saved: {HYPER_PATH}')


Saved: ../data/processed/hypermobility.csv


## 3. Merge Beighton scores into combined_data


In [52]:
survey = pd.read_csv(SURVEY_PATH)

survey['_name_key'] = survey['name'].str.strip().str.lower()
df['_name_key']     = df['name'].str.strip().str.lower()

score_map = df[df['_name_key'].notna()].set_index('_name_key')['beighton_score']
survey['beighton_score'] = survey['_name_key'].map(score_map)
survey = survey.drop(columns=['_name_key'])

matched   = survey['beighton_score'].notna().sum()
unmatched = survey['beighton_score'].isna().sum()
print(f'Matched:   {matched}')
print(f'Unmatched: {unmatched}')
survey[['name', 'age_group', 'beighton_score']]


Matched:   25
Unmatched: 0


,name,age_group,beighton_score
0,P16,30-39,1
1,P03,18-29,5
2,P24,50-59,0
3,P22,50-59,1
4,P06,18-29,8
5,P12,18-29,4
6,P04,40-49,5
7,P19,18-29,6
8,P21,18-29,5
9,P05,18-29,3


## 4. Classify hypermobility by age-adjusted threshold


In [53]:
def classify_hypermobile(row):
    score = row['beighton_score']
    if pd.isna(score):
        return pd.NA
    lower_age = int(str(row['age_group']).split('-')[0])
    if lower_age < 18:
        threshold = 6
    elif lower_age >= 50:
        threshold = 4
    else:
        threshold = 5
    return 'Yes' if score >= threshold else 'No'

survey['hypermobility'] = survey.apply(classify_hypermobile, axis=1)

print('Hypermobility classification:')
print(survey['hypermobility'].value_counts(dropna=False))
survey[['name', 'age_group', 'beighton_score', 'hypermobility']]


Hypermobility classification:
hypermobility
No     15
Yes    10
Name: count, dtype: int64


,name,age_group,beighton_score,hypermobility
0,P16,30-39,1,No
1,P03,18-29,5,Yes
2,P24,50-59,0,No
3,P22,50-59,1,No
4,P06,18-29,8,Yes
5,P12,18-29,4,No
6,P04,40-49,5,Yes
7,P19,18-29,6,Yes
8,P21,18-29,5,Yes
9,P05,18-29,3,No


## 5. Save to combined_data.csv


In [54]:
base_cols  = [c for c in survey.columns if not c.startswith('hypermobil') and c != 'beighton_score']
final_cols = base_cols + ['hypermobility']

survey[final_cols].to_csv(SURVEY_PATH, index=False)
print(f'Saved: {SURVEY_PATH}')
print(f'Shape: {survey[final_cols].shape}')


Saved: ../data/processed/combined_data.csv
Shape: (25, 18)
